In [4]:
import numpy as np
import math
from PIL import Image
from typing import Union, Any


# Define custom types for type hinting
Saliency = np.ndarray
RGBImage = Image.Image
Cmap = Any

def represent_heatmap(saliency: Saliency, cmap: Union[None, Cmap] = None) -> RGBImage:
    if cmap is None: #si on a pas de color map, on fait du niveau de gris
        grayscale_array = (saliency * 255).astype(np.uint8) # on passe de 0 à 1 de 0 à 255
        gray_img = Image.fromarray(grayscale_array, mode="L") #mode = L pour le  niveau de gris, sur un 1 seul canal
        return gray_img.convert("RGB") #on doit renvoyer une RGBImage donc on convertit en RGB
    else:
        colored_array = cmap(saliency) # on applique les couleurs
        colored_array = colored_array[:, :, :3] * 255 # on garde toutes les lignes et colonnes RGB jusqu'a l'index 2, donc pas la transparence, juste RGB
        colored_array = colored_array.astype(np.uint8) #on transforme en int
        heatmap_img = Image.fromarray(colored_array, mode="RGB")
        return heatmap_img
    
def represent_heatmap_overlaid(saliency: Saliency, image: RGBImage, cmap: Union[None, Cmap] = None) -> RGBImage:
    # On commence par générer la carte de chaleur simple en réutilisant la fonction que l'on vient de créer au-dessus
    heatmap = represent_heatmap(saliency, cmap)
    
    # Pour fusionner deux images, elles doivent avoir exactement la même taille (en pixels)
    if heatmap.size != image.size: # On vérifie si la taille diffère
        # Si c'est le cas, on redimensionne la carte de chaleur pour l'adapter à l'image d'origine (en utilisant l'algorithme LANCZOS pour une bonne qualité)
        heatmap = heatmap.resize(image.size, Image.Resampling.LANCZOS)
        
    # On s'assure par précaution que l'image de fond (la photo) est bien encodée en format RGB classique
    image_rgb = image.convert("RGB")
    
    # Application de la formule du TD : Combine(Heat(x, y), Img(x,y)). On utilise la fonction 'blend' de Pillow avec 50% de transparence (alpha=0.5)
    return Image.blend(image_rgb, heatmap, alpha=0.5) 


def represent_isolines(saliency: Saliency, cmap: Union[None, Cmap] = None) -> RGBImage:
    # On récupère les dimensions de la matrice de saillance (hauteur et largeur en pixels)
    height, width = saliency.shape 
    # On fixe une résolution d'affichage (Dots Per Inch) pour le graphique Matplotlib
    dpi = 100 
    # On crée une figure et un graphique vides ayant exactement les mêmes dimensions en pixels que notre matrice
    fig, ax = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi) 
    
    # On ajuste les marges de la figure pour qu'elles soient à zéro (le graphique prendra 100% de l'espace)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0) 
    # On cache les axes (les numéros, le cadre, les graduations)
    ax.axis('off') 
    # On peint l'arrière-plan de la fenêtre globale de la figure en noir
    fig.patch.set_facecolor('black') 
    # On peint l'arrière-plan du graphique lui-même en noir
    ax.set_facecolor('black') 
    
    if cmap is None: # Si aucune palette n'a été spécifiée
        # On trace les contours (isolignes) en niveaux de gris. 'origin=upper' indique que le point (0,0) est en haut à gauche, comme pour les images
        ax.contour(saliency, cmap='gray', origin='upper') 
    else: # Si une palette a été fournie
        # On trace les contours en utilisant les couleurs de la palette
        ax.contour(saliency, cmap=cmap, origin='upper') 
        
    # On prépare un espace en mémoire vive (un "buffer") pour capturer l'image générée sans avoir à créer de fichier physique sur ton ordinateur
    buf = io.BytesIO() 
    # On sauvegarde le graphique dans ce buffer au format PNG, en forçant explicitement la couleur de fond noire
    fig.savefig(buf, format='png', facecolor='black') 
    # On rembobine le curseur de lecture du buffer à la position zéro pour pouvoir le lire juste après
    buf.seek(0) 
    # On ferme la figure Matplotlib pour vider la mémoire (indispensable quand on génère beaucoup d'images à la suite)
    plt.close(fig) 
    
    # On utilise Pillow pour ouvrir l'image PNG stockée dans la mémoire, on la convertit en RGB, et on la retourne
    return Image.open(buf).convert("RGB") 


def represent_isolines_superimposed(saliency: Saliency, image: RGBImage, cmap: Union[None, Cmap] = None) -> RGBImage:
    # Logique très similaire à la fonction précédente : on récupère les dimensions de l'image de fond
    width, height = image.size 
    dpi = 100 # On définit la résolution
    # On crée une figure Matplotlib adaptée à la taille de l'image
    fig, ax = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi) 
    
    # On supprime toutes les marges
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0) 
    # On masque les axes
    ax.axis('off') 
    
    # ÉTAPE CLÉ : Au lieu d'avoir un fond noir, on affiche directement la photo d'origine dans le graphique
    ax.imshow(image) 
    
    if cmap is None: # Si aucune palette n'est définie
        # On superpose les lignes de contour grises par-dessus la photo
        ax.contour(saliency, cmap='gray', origin='upper') 
    else: # Sinon
        # On superpose les lignes de contour colorées
        ax.contour(saliency, cmap=cmap, origin='upper') 
        
    # On crée à nouveau un espace en mémoire (buffer)
    buf = io.BytesIO() 
    # On enregistre le graphique combiné (Photo + Contours) au format PNG dans la mémoire
    fig.savefig(buf, format='png') 
    # On repositionne le curseur au début du fichier
    buf.seek(0) 
    # On ferme la figure pour ne pas surcharger la RAM
    plt.close(fig) 
    
    # On charge le résultat avec Pillow, on le convertit en RGB et on retourne l'image finale
    return Image.open(buf).convert("RGB") 


def represent_hard_selection(saliency: Saliency, image: RGBImage, threshold: float) -> RGBImage:
    # On convertit l'image Pillow de la photo en une matrice NumPy pour pouvoir faire de l'algèbre dessus
    img_array = np.array(image.convert("RGB")) 
    
    # On applique une condition mathématique : on crée une grille de booléens (Vrai/Faux) où chaque pixel vaut Vrai s'il est au-dessus du seuil (treshold)
    mask = saliency >= threshold 
    
    # Le masque ci-dessus est en 2D (il a une hauteur et une largeur). On l'empile 3 fois sur lui-même pour créer un masque 3D qui s'appliquera aux 3 canaux Rouge, Vert, Bleu
    mask_3d = np.stack([mask] * 3, axis=-1) 
    
    # On applique la formule de la Hard Selection du TD : si le masque est Vrai, on garde la couleur de la photo, sinon on met 0 (couleur noire)
    # L'utilisation de "astype(np.uint8)" assure que le résultat final est bien codé en pixels de 8 bits
    selected_array = np.where(mask_3d, img_array, 0).astype(np.uint8) 
    
    # On retransforme la matrice filtrée en véritable image Pillow (en RGB) et on la retourne
    return Image.fromarray(selected_array, mode="RGB") 


def represent_soft_selection(saliency: Saliency, image: RGBImage, threshold: float) -> RGBImage:
    # On convertit la photo en matrice NumPy, puis on force ses valeurs en "float32" (nombres à virgule) pour éviter des erreurs lors de la multiplication
    img_array = np.array(image.convert("RGB")).astype(np.float32) 
    
    # Par précaution, on s'assure qu'aucune valeur de la saillance ne dépasse l'intervalle strict [0.0, 1.0] grâce à np.clip
    normalized_saliency = np.clip(saliency, 0, 1) 
    # Comme précédemment, on transforme cette matrice 2D de poids en matrice 3D pour l'appliquer sur les 3 canaux de couleur
    sal_3d = np.stack([normalized_saliency] * 3, axis=-1) 
    
    # Formule de la Soft Selection du TD : on multiplie chaque pixel de la photo par son poids de saillance (cela va assombrir les zones peu regardées)
    # On convertit ensuite le résultat flottant en entier 8 bits pour l'image
    soft_array = (img_array * sal_3d).astype(np.uint8) 
    
    # On génère l'image finale avec Pillow et on la retourne
    return Image.fromarray(soft_array, mode="RGB")